# Bayesian Monte Carlo k-NN on two moons

This notebook reproduces the two-moons experiment and renders the model-averaged probability surface.

The automatic ensemble starts at 20 models and doubles up to 640 models.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from bayesian_knn import BayesianKNNClassifier

In [ ]:
X, y = make_moons(
    n_samples=600,
    noise=0.24,
    random_state=7,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=7,
)

assert X.shape == (600, 2)
assert X_train.shape == (420, 2)
assert X_test.shape == (180, 2)
print(f"train shape: {X_train.shape}; test shape: {X_test.shape}")

The current estimator uses `tolerance` for the requested `mc_tolerance` setting and has a fixed 20-model automatic starting batch. The two moons are already in a common two-dimensional scale, so no standardization is applied.

In [ ]:
model = BayesianKNNClassifier(
    n_estimators="auto",
    max_estimators=640,
    tolerance=0.01,
    convergence_metric="median",
    convergence_size=256,
    cv=5,
    max_neighbors=None,
    weights="distance",
    n_jobs=-1,
    random_state=12,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
model_draws = model.get_model_draws()
model_weights = np.asarray([draw["posterior_weight"] for draw in model_draws])
effective_models = 1.0 / np.sum(model_weights**2)
largest_weight = float(model_weights.max())

print(f"test accuracy: {test_accuracy:.3f}")
print(f"estimators used: {model.n_estimators_}")
print(f"converged: {model.converged_}")
print(f"effective weighted models: {effective_models:.2f}")
print(f"largest model weight: {largest_weight:.4f}")

In [ ]:
print("Convergence history")
print("20 estimators: initial ensemble")
for entry in model.convergence_history_:
    print(
        f"{int(entry['n_estimators'])} estimators: "
        f"median probability change = {entry['median_absolute_change']:.6f}"
    )

In [ ]:
padding = 0.55
x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 350),
    np.linspace(y_min, y_max, 350),
)
grid = np.c_[xx.ravel(), yy.ravel()]
class_1_probability = model.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6.5))
heat = ax.contourf(
    xx,
    yy,
    class_1_probability,
    levels=np.linspace(0, 1, 41),
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    alpha=0.90,
)
ax.contour(xx, yy, class_1_probability, levels=[0.5], linewidths=2, colors="black")
ax.scatter(
    X_train[y_train == 0, 0], X_train[y_train == 0, 1],
    s=24, c="#2166ac", edgecolors="white", linewidths=0.45, label="Class 0",
)
ax.scatter(
    X_train[y_train == 1, 0], X_train[y_train == 1, 1],
    s=24, c="#b2182b", edgecolors="white", linewidths=0.45, label="Class 1",
)
colorbar = fig.colorbar(heat, ax=ax)
colorbar.set_label("Predicted probability of class 1\nblue = class 0, red = class 1")
ax.set_title(
    "Bayesian Monte Carlo k-NN on the two-moons dataset\n"
    f"{model.n_estimators_} estimators, 5-fold CV, test accuracy = {test_accuracy:.3f}"
)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(loc="upper right")
fig.tight_layout()

output_path = Path("moons_bayesian_knn_probability_heatmap.png")
fig.savefig(output_path, dpi=180, bbox_inches="tight")
print(f"saved {output_path}")
plt.show()

The heat map is the Bayesian model-averaged estimate of `P(Y=1 | x, D)`. The black contour is the model-averaged 0.5 decision boundary; uncertainty should be higher in the overlap and sparsely observed regions.